# EX_02 — Embeddings con Transformers (ejercicios)

**Notebook de referencia:** `notebook/02_Embeddings_Transformers.ipynb`

**Tiempo orientativo:** ~30 minutos.


## Actividad 1 — Mean pooling

Con `AutoTokenizer` + `AutoModel`, obtén **last_hidden_state** para una frase y calcula el embedding de frase como media sobre tokens (excluyendo padding).


In [1]:
from transformers import AutoTokenizer, AutoModel
import torch

text = "Transformers build contextual embeddings."
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

inputs = tokenizer(text, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)

# Mean pooling ignorando el padding mediante la máscara de atención
mask = inputs['attention_mask'].unsqueeze(-1).float()
sum_embeds = torch.sum(outputs.last_hidden_state * mask, dim=1)
sum_mask = torch.clamp(mask.sum(dim=1), min=1e-9)
mean_pooled = sum_embeds / sum_mask

print("Shape del embedding final:", mean_pooled.shape)

/opt/anaconda3/envs/IA_MUBA/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Shape del embedding final: torch.Size([1, 768])


## Actividad 2 — `sentence-transformers`

Usa `SentenceTransformer` para embedder dos frases y calcula similitud coseno. Comenta brevemente (en inglés en un comentario) por qué suele ser mejor que mean-pooling manual de BERT base.


In [2]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')
embs = model.encode(["I like dogs", "I am fond of canines"])

sim = cosine_similarity([embs[0]], [embs[1]])
print(f"Similitud: {sim[0][0]:.4f}")

# Comentario: sentence-transformers is fine-tuned specifically using Siamese networks 
# to output semantically meaningful sentence embeddings, whereas standard BERT's mean pooling yields poor sentence similarities.

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
2026-05-25 18:07:57.092013: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/opt/anaconda3/envs/IA_MUBA/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/

Similitud: 0.7940


## Actividad 3 — Paráfrasis

Escribe dos paráfrasis de una misma idea y muestra que sus embeddings (sentence-transformers) tienen **mayor** similitud entre sí que con una frase de tema distinto.


In [3]:
# TODO: three strings, 2 paraphrase + 1 unrelated, print cosine sims
text1 = "El clima de hoy es perfecto para ir a la playa."
text2 = "Hace un día soleado ideal para disfrutar del mar."
text3 = "Las tasas de interés de los bancos han subido."

embs = model.encode([text1, text2, text3])

sim_paraphrase = cosine_similarity([embs[0]], [embs[1]])[0][0]
sim_unrelated = cosine_similarity([embs[0]], [embs[2]])[0][0]

print(f"Similitud Paráfrasis: {sim_paraphrase:.4f}")
print(f"Similitud No Relacionada: {sim_unrelated:.4f}")


Similitud Paráfrasis: 0.4669
Similitud No Relacionada: 0.4021
